# Model Performance Evaluation

Plots for measuring how well the PPO portfolio agent performed on the test set, benchmarked against Equal-Weight Buy & Hold.

Since this is a portfolio-allocation RL system (not a classifier), there is no single "accuracy" score. The closest, meaningful measures are:
- **Win rate** — % of trading days with a positive portfolio return (directional accuracy)
- **Risk-adjusted return metrics** — Sharpe, Sortino, Calmar (return achieved per unit of risk)
- **Drawdown** — how far the portfolio ever fell from its peak
- **Head-to-head comparison** vs the Equal-Weight benchmark

Data source: `results/nb_run_20260728_233032/` (test trajectory + benchmark comparison already produced by the backtest).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.metrics import PortfolioMetrics

RESULTS_DIR = PROJECT_ROOT / "results" / "nb_run_20260728_233032"

plt.style.use("seaborn-v0_8-darkgrid")
%matplotlib inline

## Load backtest results

In [ ]:
trajectory = pd.read_csv(RESULTS_DIR / "test_trajectory.csv", parse_dates=["date"])
comparison = pd.read_csv(RESULTS_DIR / "test_vs_benchmark.csv", index_col=0)

portfolio_values = trajectory["portfolio_value"].to_numpy()
portfolio_returns = trajectory["portfolio_return"].to_numpy()
dates = trajectory["date"]

trajectory.head()

## Portfolio value & drawdown

In [ ]:
peak = np.maximum.accumulate(portfolio_values)
drawdown = (peak - portfolio_values) / (peak + 1e-10)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), height_ratios=[3, 1], sharex=True)

axes[0].plot(dates, portfolio_values, label="PPO Agent", linewidth=2, color="#2196F3")
axes[0].set_title("Portfolio Value Over Time", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Portfolio Value (LKR)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].fill_between(dates, drawdown, alpha=0.4, color="#F44336")
axes[1].set_ylabel("Drawdown")
axes[1].set_xlabel("Date")
axes[1].invert_yaxis()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## PPO vs Equal-Weight benchmark

In [ ]:
plot_metrics = ["sharpe_ratio", "sortino_ratio", "annualized_return", "annualized_volatility"]
subset = comparison.loc[plot_metrics]

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(subset))
width = 0.35

ax.bar(x - width / 2, subset.iloc[:, 0], width, label=subset.columns[0], color="#2196F3")
ax.bar(x + width / 2, subset.iloc[:, 1], width, label=subset.columns[1], color="#FF9800")

ax.set_xticks(x)
ax.set_xticklabels(subset.index, rotation=20)
ax.set_title("PPO Agent vs Equal-Weight Benchmark", fontsize=14, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

comparison

## Rolling Sharpe ratio (20-day window)

In [ ]:
metrics_calc = PortfolioMetrics(risk_free_rate=0.06)
window = 20

returns_series = pd.Series(portfolio_returns)
rolling_sharpe = returns_series.rolling(window).apply(
    lambda r: metrics_calc.sharpe_ratio(r.to_numpy()), raw=False
)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(dates, rolling_sharpe, color="#4CAF50", linewidth=1.5)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title(f"Rolling {window}-Day Sharpe Ratio", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Sharpe Ratio (annualized)")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Win rate (directional accuracy)

Percentage of trading days the portfolio return was positive — the closest analogue to "accuracy" for a trading strategy.

In [ ]:
win_rate = metrics_calc.win_rate(portfolio_returns)
loss_rate = 1 - win_rate

fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(
    [win_rate, loss_rate],
    labels=[f"Positive days\n{win_rate:.1%}", f"Negative days\n{loss_rate:.1%}"],
    colors=["#4CAF50", "#F44336"],
    autopct="%1.1f%%",
    startangle=90,
)
ax.set_title("Win Rate", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Win rate: {win_rate:.2%}  ({int(win_rate * len(portfolio_returns))} / {len(portfolio_returns)} days)")

## Portfolio weight allocation over time

In [ ]:
weight_cols = [c for c in trajectory.columns if c.startswith("w_")]
asset_names = [c.replace("w_", "") for c in weight_cols]

fig, ax = plt.subplots(figsize=(14, 6))
ax.stackplot(
    dates,
    *[trajectory[c] for c in weight_cols],
    labels=asset_names,
    alpha=0.85,
)
ax.set_title("Portfolio Weight Allocation Over Time", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Weight")
ax.set_ylim(0, 1)
ax.legend(loc="upper left", ncol=min(5, len(weight_cols)), fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()